In [188]:
import numpy as np
from scipy.stats import pearsonr
from itertools import combinations


In [189]:
# wczytywanie danych
dane = []
with open("dobor_zmiennych.txt", 'r') as fi:
    for line in fi:
        if line.split():
            line = [float(x) for x in line.split()]
            dane.append(line)


l = len(dane[0])

zmienne = ['X' + str(i) for i in range(1, l)]
print(l, zmienne)
Y = []
for i in range(1, l):
    zmienne[i-1] = []
for item in dane:
    Y.append(item[0])
    for i in range(1, l):
        zmienne[i-1].append(item[i])

for i in range(1, l):
    print(zmienne[i-1]) 



4 ['X1', 'X2', 'X3']
[5308.8, 5448.2, 5259.1, 5586.6, 6052.3, 6465.8, 7928.0, 1194.8, 19782.0, 33059.0, 73286.0, 575.6, 1265.2, 1347.8, 1590.1, 1733.2, 2132.8, 2761.4, 3361.0, 4005.1, 4590.5, 3981.5]
[389.0, 393.0, 384.0, 353.0, 351.0, 348.0, 350.0, 349.0, 353.0, 374.0, 378.0, 390.0, 387.0, 389.0, 373.0, 362.0, 338.0, 347.0, 343.0, 353.0, 342.0, 344.0]
[21578.0, 21928.0, 19066.0, 18463.0, 18418.0, 18289.0, 18492.0, 17811.0, 17259.0, 15499.0, 14940.0, 13816.0, 12416.0, 11866.0, 11856.0, 11721.0, 11715.0, 11605.0, 11520.0, 11524.0, 11462.0, 11459.0]


In [190]:
# usuwanie guasi-stałych, tworzenie wektora i macierzy korelacji (gdy podane są dane)
def usun_quasistale(zmienne, v):
    zmienne_z = []
    for i in range(len(zmienne)):
        X = zmienne[i]
        if abs(np.std(X)/np.mean(X)) > v:
            zmienne_z.append(X)      
    return zmienne_z

def utworz_wektor_korelacji(zmienne, Y):
    wektor_korelacji = []
    for X in zmienne:
        wektor_korelacji.append(abs(pearsonr(X, Y)[0]))
    return wektor_korelacji

def utworz_macierz_korelacji(zmienne):
    macierz_korelacji = []
    for X in zmienne:
        wiersz = []
        for Z in zmienne:
            wiersz.append(abs(pearsonr(X, Z)[0]))
        macierz_korelacji.append(wiersz)
    return macierz_korelacji

In [191]:
# wczytywanie wektora i macierzy korelacji ( gdy mamy tylko wektor i macierz bez danych źródłowych)
wektor = []  # wczytuje macierz korelacji jednocześnie biorąc wartośc bezwzględną
with open("przyklad2_wektor.txt", 'r') as fi:
    for line in fi:
        if line.split():
            line = [abs(float(x)) for x in line.split()]
            wektor.append(line)
wektor_korelacji_w = wektor[0]



macierz_korelacji_w = [] # wczytuje macierz korelacji jednocześnie biorąc wartośc bezwzględną
with open("przyklad2_macierz.txt", 'r') as fi:
    for line in fi:
        if line.split():
            line = [abs(float(x)) for x in line.split()]
            macierz_korelacji_w.append(line)

macierz_korelacji_P = [] # wczytuje macierz korelacji 
with open("przyklad2_macierz.txt", 'r') as fi:
    for line in fi:
        if line.split():
            line = [(float(x)) for x in line.split()]
            macierz_korelacji_P.append(line)

print(wektor_korelacji_w)
print(macierz_korelacji_w)
print(macierz_korelacji_P)


[0.55, 0.87, 0.85, 0.47, 0.45, 0.65]
[[1.0, 0.34, 0.56, 0.21, 0.98, 0.4], [0.34, 1.0, 0.55, 0.45, 0.8, 0.5], [0.56, 0.55, 1.0, 0.32, 0.4, 0.11], [0.21, 0.45, 0.32, 1.0, 0.2, 0.5], [0.98, 0.8, 0.4, 0.2, 1.0, 0.8], [0.4, 0.5, 0.11, 0.5, 0.8, 1.0]]
[[1.0, 0.34, 0.56, 0.21, 0.98, -0.4], [0.34, 1.0, 0.55, 0.45, 0.8, 0.5], [0.56, 0.55, 1.0, -0.32, 0.4, 0.11], [0.21, 0.45, -0.32, 1.0, -0.2, 0.5], [0.98, 0.8, 0.4, -0.2, 1.0, 0.8], [-0.4, 0.5, 0.11, 0.5, 0.8, 1.0]]


In [192]:
def eliminacja_nadmiarowych_zmiennych(wektor_korelacji, macierz_korelacji, r_kryt):
    
    wektor_korelacji = np.asarray(wektor_korelacji)
    R = np.asarray(macierz_korelacji)

    # KROK 1: eliminacja zmiennych słabo skorelowanych z Y
    kandydaci = [
        i for i, r in enumerate(wektor_korelacji)
        if abs(r) >= r_kryt
    ]

    wybrane = []

    # KROK 2–4: wybór zmiennej i usuwanie silnie skorelowanych
    while kandydaci:
        # zmienna o największej korelacji z Y
        najlepsza = max(kandydaci, key=lambda i: abs(wektor_korelacji[i]))
        wybrane.append(najlepsza)
        print('wybrane', wybrane)
        # pozostają tylko zmienne słabo skorelowane z wybraną
        kandydaci = [
            i for i in kandydaci
            if i != najlepsza and R[najlepsza, i] <= r_kryt
        ]
        print('kandydaci', kandydaci)
    # numeracja od 1 (X₁, X₂, …)
    return [i + 1 for i in wybrane]

In [193]:
zmienne_z = usun_quasistale(zmienne, 0.01)
eliminacja_nadmiarowych_zmiennych(utworz_wektor_korelacji(zmienne_z, Y), utworz_macierz_korelacji(zmienne_z), 0.5)

wybrane [2]
kandydaci []


[3]

In [194]:
eliminacja_nadmiarowych_zmiennych(wektor_korelacji_w, macierz_korelacji_w, 0.5)

wybrane [1]
kandydaci [0, 5]
wybrane [1, 5]
kandydaci [0]
wybrane [1, 5, 0]
kandydaci []


[2, 6, 1]

In [195]:
def metoda_pawlowskiego(macierz_korelacji, k):
    R = np.asarray(macierz_korelacji)
    n = R.shape[0]

    max_det = -np.inf
    najlepszy_zestaw = None

    # wszystkie kombinacje k zmiennych
    for idx in combinations(range(n), k):
        podmacierz = R[np.ix_(idx, idx)]
        det = np.linalg.det(podmacierz)
        print(podmacierz, det)

        if det > max_det:
            max_det = det
            najlepszy_zestaw = idx

    # numeracja od 1 (X1, X2, ...)
    return [i + 1 for i in najlepszy_zestaw], max_det


In [196]:
# metoda_pawlowskiego(macierz_korelacji_P, 4)

In [197]:
macierz = utworz_macierz_korelacji(zmienne_z)
metoda_pawlowskiego(macierz,2)


[[1.         0.15812995]
 [0.15812995 1.        ]] 0.9749949180647407
[[1.         0.10401199]
 [0.10401199 1.        ]] 0.9891815054074338
[[1.         0.28633901]
 [0.28633901 1.        ]] 0.9180099696786151


([1, 3], np.float64(0.9891815054074338))